In [12]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
%matplotlib inline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import roc_auc_score, f1_score, recall_score, precision_score
from scipy.stats.mstats import winsorize
from imblearn.over_sampling import SMOTE

import warnings
warnings.filterwarnings('ignore')

In [13]:
heart_disease_dataset = pd.read_csv("../data/processed/heart_processed.csv")

In [14]:
heart_disease_dataset.head()

,Age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,52,1,0,125,212,0,1,168,0,1.0,2,2,3,0
1,53,1,0,140,203,1,0,155,1,3.1,0,0,3,0
2,70,1,0,145,174,0,1,125,1,2.6,0,0,3,0
3,61,1,0,148,203,0,1,161,0,0.0,2,1,3,0
4,62,0,0,138,294,1,1,106,0,1.9,1,3,2,0


In [15]:
# Changing data types of categorical features
heart_disease_dataset['sex'] = heart_disease_dataset['sex'].astype('category')
heart_disease_dataset['fbs'] = heart_disease_dataset['fbs'].astype('category')
heart_disease_dataset['exang'] = heart_disease_dataset['exang'].astype('category')
heart_disease_dataset['target'] = heart_disease_dataset['target'].astype('category')
heart_disease_dataset['cp'] = heart_disease_dataset['cp'].astype('category')
heart_disease_dataset['restecg'] = heart_disease_dataset['restecg'].astype('category')
heart_disease_dataset['slope'] = heart_disease_dataset['slope'].astype('category')
heart_disease_dataset['thal'] = heart_disease_dataset['thal'].astype('category')
heart_disease_dataset['ca'] = heart_disease_dataset['ca'].astype('category')

In [16]:
heart_disease_dataset.corr()

,Age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
Age,1.000000,-0.103240,-0.071966,0.271121,0.219823,0.121243,-0.132696,-0.390227,0.088163,0.208137,-0.169105,0.271551,0.072297,-0.229324
sex,-0.103240,1.000000,-0.041119,-0.078974,-0.198258,0.027200,-0.055117,-0.049365,0.139157,0.084687,-0.026666,0.111729,0.198424,-0.279501
cp,-0.071966,-0.041119,1.000000,0.038177,-0.081641,0.079294,0.043581,0.306839,-0.401513,-0.174733,0.131633,-0.176206,-0.163341,0.434854
trestbps,0.271121,-0.078974,0.038177,1.000000,0.127977,0.181767,-0.123794,-0.039264,0.061197,0.187434,-0.120445,0.104554,0.059276,-0.138772
chol,0.219823,-0.198258,-0.081641,0.127977,1.000000,0.026917,-0.147410,-0.021772,0.067382,0.064880,-0.014248,0.074259,0.100244,-0.099966
fbs,0.121243,0.027200,0.079294,0.181767,0.026917,1.000000,-0.104051,-0.008866,0.049261,0.010859,-0.061902,0.137156,-0.042177,-0.041164
restecg,-0.132696,-0.055117,0.043581,-0.123794,-0.147410,-0.104051,1.000000,0.048411,-0.065606,-0.050114,0.086086,-0.078072,-0.020504,0.134468
thalach,-0.390227,-0.049365,0.306839,-0.039264,-0.021772,-0.008866,0.048411,1.000000,-0.380281,-0.349796,0.395308,-0.207888,-0.098068,0.422895
exang,0.088163,0.139157,-0.401513,0.061197,0.067382,0.049261,-0.065606,-0.380281,1.000000,0.310844,-0.267335,0.107849,0.197201,-0.438029
oldpeak,0.208137,0.084687,-0.174733,0.187434,0.064880,0.010859,-0.050114,-0.349796,0.310844,1.000000,-0.575189,0.221816,0.202672,-0.438441


In [17]:
#log-transforming highly skewed continous features
for col in ["oldpeak", "chol"]:
    heart_disease_dataset[col] = np.log1p(heart_disease_dataset[col])

In [18]:
#VIF Analysis
from statsmodels.stats.outliers_influence import variance_inflation_factor

def calculate_vif(df):
    vif = pd.DataFrame()
    vif['Features'] = df.columns
    vif['VIF'] = [variance_inflation_factor(df.values, i) for i in range(df.shape[1])]
    return vif

def drop_high_vif_features(df, threshold=5):
    dropped_features = []
    df = df.copy()
    while True:
        vif = calculate_vif(df)
        max_vif = vif["VIF"].max()
        if max_vif > threshold:
            drop_feature = vif.sort_values("VIF", ascending=False)["Features"].iloc[0]
            print(f"Dropping feature '{drop_feature}' with VIF '{max_vif:.3f}")
            df = df.drop(columns=[drop_feature])
            dropped_features.append(drop_feature)
            print("Remaining features\n", df.columns.tolist())
        else:
            break
    return df, dropped_features
X = heart_disease_dataset.drop(columns=['target'])
X_reduced, dropped = drop_high_vif_features(X, threshold=10.0)
print("Dropped features:", dropped)
print("Final VIFs:\n", calculate_vif(X_reduced))

Dropping feature 'chol' with VIF '186.220
Remaining features
 ['Age', 'sex', 'cp', 'trestbps', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal']
Dropping feature 'trestbps' with VIF '56.819
Remaining features
 ['Age', 'sex', 'cp', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal']
Dropping feature 'thalach' with VIF '29.094
Remaining features
 ['Age', 'sex', 'cp', 'fbs', 'restecg', 'exang', 'oldpeak', 'slope', 'ca', 'thal']
Dropping feature 'Age' with VIF '18.797
Remaining features
 ['sex', 'cp', 'fbs', 'restecg', 'exang', 'oldpeak', 'slope', 'ca', 'thal']
Dropping feature 'thal' with VIF '10.729
Remaining features
 ['sex', 'cp', 'fbs', 'restecg', 'exang', 'oldpeak', 'slope', 'ca']
Dropped features: ['chol', 'trestbps', 'thalach', 'Age', 'thal']
Final VIFs:
   Features       VIF
0      sex  3.153505
1       cp  2.102299
2      fbs  1.223841
3  restecg  1.962503
4    exang  1.905551
5  oldpeak  2.428587
6    slope  3.681167
7       ca  1.696329

In [19]:
# Dividing dataset into exploratory features and target
X = heart_disease_dataset.drop("target", axis=1)
y = heart_disease_dataset["target"]

In [20]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [21]:
scaler = StandardScaler()
X_train_norm = scaler.fit_transform(X_train)
X_test_norm = scaler.transform(X_test)

In [22]:
# Performing GridSeearchCV for ElasticNet Model
param_grid = {
    'C': [0.01, 0.1, 1, 10],
    'l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9]
}

elasticnet_lr = LogisticRegression(penalty='elasticnet', solver='saga', max_iter=5000, random_state=42)
grid_search = GridSearchCV(elasticnet_lr, param_grid, cv=5, scoring='recall')
grid_search.fit(X_train_norm, y_train)

print("Best parameters:", grid_search.best_params_)
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test_norm)

Best parameters: {'C': 0.01, 'l1_ratio': 0.1}


In [23]:
# Performing GridSearchCV for Lasso-Regression Model
param_grid = {
    'C': [0.01, 0.1, 1, 10],
    'solver': ['liblinear', 'saga']  # Optional: tune solver
}

lasso_lr = LogisticRegression(penalty='l1', max_iter=5000, random_state=42)
grid_search_lasso = GridSearchCV(lasso_lr, param_grid, cv=5, scoring='recall')
grid_search_lasso.fit(X_train_norm, y_train)

print("Best parameters:", grid_search_lasso.best_params_)
best_lasso = grid_search_lasso.best_estimator_
y_pred_lasso = best_lasso.predict(X_test_norm)

Best parameters: {'C': 0.1, 'solver': 'saga'}


In [24]:
param_grid = {
    'C': [0.01, 0.1, 1, 10],
    'solver': ['lbfgs', 'saga']  # Both support L2 penalty
}

ridge_lr = LogisticRegression(penalty='l2', max_iter=5000, random_state=42)
grid_search_ridge = GridSearchCV(ridge_lr, param_grid, cv=5, scoring='recall')
grid_search_ridge.fit(X_train_norm, y_train)

print("Best parameters:", grid_search_ridge.best_params_)
best_ridge = grid_search_ridge.best_estimator_
y_pred_ridge = best_ridge.predict(X_test_norm)

Best parameters: {'C': 0.01, 'solver': 'lbfgs'}


In [25]:
# GridSearchCV for Random Forest Classifier
param_grid_rf = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 5, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'class_weight': ['balanced', None]
}

rf = RandomForestClassifier(random_state=42)
grid_search_rf = GridSearchCV(rf, param_grid_rf, cv=5, scoring='recall')
grid_search_rf.fit(X_train_norm, y_train)

print("Best parameters for Random Forest:", grid_search_rf.best_params_)
best_rf = grid_search_rf.best_estimator_
y_pred_rf = best_rf.predict(X_test_norm)

Best parameters for Random Forest: {'class_weight': 'balanced', 'max_depth': 10, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}


In [26]:
lasso_lr = LogisticRegression(penalty='l1', max_iter=5000, C=0.1, solver='saga', random_state=42)
lasso_lr.fit(X_train_norm , y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'l1'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",0.1
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multiclass` p

In [27]:
# Evaluating metrics for Lasso Regression Model
y_pred_lasso = lasso_lr.predict(X_test_norm)
y_proba_lasso = lasso_lr.predict_proba(X_test_norm)[:, 1]

roc_auc_lasso = roc_auc_score(y_test, y_proba_lasso)
recall_lasso = recall_score(y_test, y_pred_lasso)
f1_lasso = f1_score(y_test, y_pred_lasso)
precision_lasso = precision_score(y_test, y_pred_lasso)

print(f"Lasso ROC-AUC: {roc_auc_lasso:.3f}")
print(f"Lasso Recall: {recall_lasso:.3f}")
print(f"Lasso F1 Score: {f1_lasso:.3f}")
print(f"Lasso Precision: {precision_lasso:.3f}")

Lasso ROC-AUC: 0.899
Lasso Recall: 0.893
Lasso F1 Score: 0.831
Lasso Precision: 0.778


In [28]:
# Traiing Elastic Net Model
elasticnet_lr = LogisticRegression(penalty='elasticnet', solver='saga', max_iter=5000, C=0.01, l1_ratio=0.1, random_state=42)
elasticnet_lr.fit(X_train_norm, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'elasticnet'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",0.01
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.1
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mult

In [29]:
# Evaluating metrics for ElasticNet Regression Model
y_pred_elasticnet = elasticnet_lr.predict(X_test_norm)
y_proba_elasticnet = elasticnet_lr.predict_proba(X_test_norm)[:, 1]

roc_auc_elasticnet = roc_auc_score(y_test, y_proba_elasticnet)
recall_elasticnet = recall_score(y_test, y_pred_elasticnet)
f1_elasticnet = f1_score(y_test, y_pred_elasticnet)
precision_elasticnet = precision_score(y_test, y_pred_elasticnet)

print(f"Elasticnet ROC-AUC: {roc_auc_elasticnet:.3f}")
print(f"Elasticnet Recall: {recall_elasticnet:.3f}")
print(f"Elasticnet F1 Score: {f1_elasticnet:.3f}")
print(f"ELasticnet Precision: {precision_elasticnet:.3f}")

Elasticnet ROC-AUC: 0.901
Elasticnet Recall: 0.906
Elasticnet F1 Score: 0.821
ELasticnet Precision: 0.750


In [30]:
# Training ridge regression model
ridge_lr = LogisticRegression(penalty='l2', C =0.01, solver='lbfgs', max_iter=5000, random_state=42)
ridge_lr.fit(X_train_norm, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'l2'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",0.01
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multiclass` 

In [31]:
# Evaluating metrics for Ridge Regression Model
y_pred_ridge = ridge_lr.predict(X_test_norm)
y_proba_ridge = ridge_lr.predict_proba(X_test_norm)[:, 1]

roc_auc_ridge = roc_auc_score(y_test, y_proba_ridge)
recall_ridge = recall_score(y_test, y_pred_ridge)
f1_ridge = f1_score(y_test, y_pred_ridge)
precision_ridge = precision_score(y_test, y_pred_ridge)

print(f"RLR ROC-AUC: {roc_auc_ridge:.3f}")
print(f"RLR Recall: {recall_ridge:.3f}")
print(f"RLR F1 Score: {f1_ridge:.3f}")
print(f"RLR Precision: {precision_ridge:.3f}")

RLR ROC-AUC: 0.902
RLR Recall: 0.913
RLR F1 Score: 0.824
RLR Precision: 0.751


### Model Selection Rationale

After evaluating Lasso, ElasticNet, and Ridge (L2) logistic regression models, I chose the **Lasso Regression** model for heart disease prediction. The Lasso model achieved a strong balance between recall (0.893), F1 score (0.831), and the highest precision (0.778) among the tested models. In medical risk prediction, while recall is important to minimize false negatives, precision is also valuable to reduce unnecessary interventions. Lasso’s performance offers a good trade-off, and its feature selection property can enhance model interpretability.